# 12. Работа с трендом и сезонностью

Рабочий ноутбук к главе книги `../chapters/12-trend-seasonality.md`: здесь только задания и код, теория и выводы — в главе. Источник — глава 16 пособия.

Графики для главы сохраняй через `save_fig(fig, "имя")` → `../chapters/img/12-имя.png`.

In [ ]:
import logging
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from etna.datasets import TSDataset
from etna.transforms import (
    LinearTrendTransform,
    TheilSenTrendTransform,
    ChangePointsTrendTransform,
    STLTransform,
    DeseasonalityTransform,
    FourierTransform,
)
from etna.models import SeasonalMovingAverageModel, NaiveModel
from etna.pipeline import Pipeline
from etna.metrics import SMAPE
from etna.analysis import plot_trend, seasonal_plot, stl_plot, acf_plot, plot_periodogram, plot_backtest

logging.getLogger("cmdstanpy").disabled = True
warnings.filterwarnings("ignore", category=FutureWarning)

DATA_DIR = "../local/data"
IMG_DIR = "../chapters/img"  # графики для главы книги
HORIZON = 14
RANDOM_SEED = 42


def save_fig(fig, name: str) -> None:
    """Сохранить график для главы книги: ../chapters/img/12-<name>.png."""
    os.makedirs(IMG_DIR, exist_ok=True)
    fig.savefig(f"{IMG_DIR}/12-{name}.png", dpi=150, bbox_inches="tight")

## Задание 1. Синтетика: разложить ряд на части

Ряд, у которого мы знаем тренд и сезонность.

### Шаг 1.1. Ряд с трендом и сезонностью

Синтетический ряд: линейный тренд + недельный узор + шум (как в главе про CatBoost).

### Шаг 1.2. Оценка тренда

`plot_trend` с `LinearTrendTransform`, `TheilSenTrendTransform`, `ChangePointsTrendTransform`; `stl_plot`.

### Шаг 1.3. Детрендирование

Модель без тренда (`SeasonalMovingAverageModel`) на сыром ряде и после `LinearTrendTransform`.

## Задание 2. Реальные юниты: какая у нас сезонность

2 юнита `10+` с 2025-01-01 — длинная история.

### Шаг 2.1. Загрузка

Прочитай `facts-by-date.csv` и `units.csv` из `DATA_DIR`, собери `TSDataset` (дневная частота, `segment` = `MaskedUnitId`). Период — с 2025-01-01, 2 юнита `10+`.

### Шаг 2.2. Сезонность

`seasonal_plot` (неделя и год), `acf_plot`, `plot_periodogram`: какие периоды есть в рядах.

### Шаг 2.3. Разложение

`stl_plot`: как менялись тренд и недельный узор за полтора года.

## Задание 3. Трансформы в конвейере

Помогают ли трансформы тренда и сезонности прогнозу.

### Шаг 3.1. Варианты конвейера

Модель без трансформов; + `LinearTrendTransform`; + `STLTransform(period=7)`; + `DeseasonalityTransform(period=7)`.

### Шаг 3.2. Backtest

Backtest всех вариантов на стандартном периоде (с 2026-04-01): SMAPE по фолдам.